In [ ]:
import numpy as np
import os
import re

In [ ]:
def smart_split(s):
    # Remove enclosing braces if any
    s = s.strip()
    if s.startswith('{') and s.endswith('}'):
        s = s[1:-1]
    # Split on either comma or whitespace
    return re.split(r'[,\s]+', s.strip())

In [ ]:
def read_dat(root, filename):
    with open(f'{root}/{filename}', 'r') as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith('*')]

    if isinstance(eval(lines[0]), str):
        lines = lines[1:]
    b = np.array([float(n) for n in smart_split(lines[3])], dtype=np.float32)

    m = int(lines[0])
    block_sizes = [int(n) for n in smart_split(lines[2])]
    block_sizes = np.abs(block_sizes)
    assert len(block_sizes) == int(lines[1])
    biases = np.cumsum(np.hstack([np.zeros(1, dtype=np.int32), block_sizes]))

    n = sum(block_sizes)
    Fs = np.zeros((n, n, m + 1), dtype=np.float32)
    for line in lines[4:]:
        num_mat, num_block, row, col, num = smart_split(line)
        num = float(num)
        if num:
            num_mat = int(num_mat)
            num_block = int(num_block) - 1
            row = int(row) - 1
            col = int(col) - 1
            bias = biases[num_block]
            Fs[bias + row, bias + col, num_mat] = num

    Fs = (Fs + np.transpose(Fs, (1, 0, 2))) / 2
    C = -Fs[..., 0]
    A = Fs[..., 1:]
    return C, A, b

In [ ]:
root = './SDPLIB'

In [ ]:
files = [f for f in os.listdir(root) if f.endswith('dat-s')]

In [ ]:
exceptions = ['maxG32.dat-s', 'maxG55.dat-s', 'maxG60.dat-s', 
              'theta4.dat-s', 'theta5.dat-s', 'theta6.dat-s', 'thetaG11.dat-s', 'thetaG51.dat-s', ]

In [ ]:
files = [f for f in files if f not in exceptions]

## we take mcp family

In [ ]:
done_files = [f for f in os.listdir(root) if f.endswith('.npz') and f.startswith('mcp')]

for filename in files:
    f = filename.split('.')[0]
    if f"{f}.npz" not in done_files:
        print(filename)
        C, A, b = read_dat(root, filename)
        print(filename, A.shape[-1], C.shape[0])
        np.savez(f'{root}/{f}.npz', C, A, b)

In [ ]:
done_files = [f for f in os.listdir(root) if f.endswith('.npz') and f.startswith('mcp')]

In [ ]:
from tqdm import tqdm

import os
import torch
from torch_geometric.data import Batch, HeteroData
from numpy.linalg import LinAlgError

from utils.evaluation import solve_sdp_cvxpy, solve_sdp_scs
from torch_geometric.utils import to_dense_adj

from cvxpy import DCPError, DGPError, DPPError, SolverError
from utils.evaluation import map_vec, mat

In [ ]:
graphs = []

for file in done_files:
    f = np.load(f'{root}/{file}')
    print(file)
    C, A, b = f['arr_0'], f['arr_1'], f['arr_2']

    X, y, dual, sol = solve_sdp_scs(C, A, b)
    assert sol['info']['status'].startswith('solved')

    print(sol['info']['status'], sol['info']['solve_time'])
    
    m = b.shape[0]
    n = C.shape[0]
    A = torch.from_numpy(A).float()
    A = A.reshape(-1, A.shape[-1]).T  # m, n**2
    A_where = torch.where(A)
    
    c2v_idx = torch.vstack(A_where)
    c2v_value = A[A_where][:, None]
    
    C = torch.from_numpy(C).float().reshape(-1)[None]
    # sparse vals obj connections
    C_where = torch.where(C)
    o2v_idx = torch.vstack(C_where)
    o2v_value = C[C_where][:, None]

    x = torch.from_numpy(X).float().reshape(-1)
    y = torch.from_numpy(y).float()
    dual = torch.from_numpy(dual).float().reshape(-1)

    data = HeteroData(
        cons={
            'num_nodes': m,
            'x': torch.empty(m, 0),
             },
        vals={
            'num_nodes': n ** 2,
            'x': torch.empty(n ** 2, 0),
        },
        obj={
            'num_nodes': 1,
            'x': torch.ones(1).float(),
             },
        cons__to__vals={'edge_index': c2v_idx,
                        'edge_attr': c2v_value},
        obj__to__vals={'edge_index': o2v_idx,
                        'edge_attr': o2v_value},
        x_solution=x,
        y_solution=y,
        dual_solution=dual,
        obj_solution=torch.tensor([sol['info']['pobj']]),
        b=torch.from_numpy(b).float(),
    )
    data.name = file.split('.')[0]
    graphs.append(data)

## training set only

In [ ]:
from torch_geometric.data import InMemoryDataset

torch.save(InMemoryDataset().collate(graphs), f'{root}/processed/train.pt')
torch.save(None, f'{root}/processed/valid.pt')
torch.save(None, f'{root}/processed/test.pt')

In [ ]:
from data.dataset import LPDataset

In [ ]:
train_set = LPDataset(root, 'train', transform=None)

In [ ]:
train_set.data.x_solution

In [ ]:
train_set.data.dual_solution